# BloodMNIST + OrganCMNIST — correctness memory: $z_{\mathrm{combined}}$ vs $H$

Cross-dataset generalization notebook (secondary to DermaMNIST).

Classifier training is **unchanged**; only the side-channel memory uses a correctness target:

$$u_t = c_t P k_t \in \mathbb{R}^{7}, \quad M^{(j)} \in \mathbb{R}^{7 \times d_k}, \quad z_{\mathrm{combined}} \in \mathbb{R}^{28}$$

For each dataset:
1. **Train** ResNet-18 + correctness memory (`train_one_seed_correctness`)
2. **Deploy** label-free memory features on ID calibration + ID test
3. **Evaluate** cal-fit logistic on $z_{\mathrm{combined}}$ vs normalized entropy $H$ (AUROC, AUPRC)

**Sample caps (match DermaMNIST):** train $=6408$, cal $=1602$, test $=2005$; probe eval uses $n=2000$ test subsample per seed.

Probe fitting uses ID calibration only; test labels are used only for offline metrics.

## 0. Setup

In [1]:
from __future__ import annotations

import json
import pickle
import shutil
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_repo_root(cwd: Path | None = None) -> Path:
    root = Path(cwd or Path.cwd()).resolve()
    if (root / "research").exists():
        return root
    if (root.parent / "research").exists():
        return root.parent
    return root


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from optimizer.associative_memory import AssociativeMemoryConfig
from research.common.clinical_datasets import ClinicalDatasetConfig, load_clinical_bundle
from research.common.clinical_training import ClinicalTrainingConfig, checkpoint_path, save_checkpoint
from research.common.correctness_clinical_training import train_one_seed_correctness
from research.common.correctness_deployment_pipeline import run_correctness_deployment_on_split
from research.common.correctness_memory_io import (
    correctness_artifacts_exist,
    correctness_checkpoint_bundle_path,
    correctness_memory_exists,
    correctness_memory_path,
    load_correctness_artifacts,
    recover_correctness_checkpoint_bundle,
)
from research.correctness_memory_fft.probe_experiments.correctness_probe_analysis import (
    cumulative_cols as z_combined_cols,
    level_cols,
)

print("imports ok")

imports ok


## Configuration

In [2]:
TASKS = ("bloodmnist", "organcmnist")
OUTPUT_DIR = REPO_ROOT / "research" / "correctness_memory_fft" / "cross_dataset"
PROBE_DIR = OUTPUT_DIR / "probe_results"
FEATURE_ROOT = OUTPUT_DIR / "probe_features"
DATA_DIR = REPO_ROOT / "data" / "clinical"
SEEDS = (42, 123, 456)

CORRECTNESS_Z_DIM = 7  # z_j in R^7; z_combined in R^{4*7}=28
NUM_LEVELS = 4

RUN_BUILD = False
FORCE_RETRAIN = False
FORCE_REDEPLOY = False
LOGISTIC_C = 1.0
RANDOM_SEED = 42

# Match DermaMNIST protocol
DERMA_TRAIN = 5000
DERMA_CAL = 1602
DERMA_TEST = 2000
TEST_SAMPLE_SIZE = 2000

FAST_MODE = False
ASSOC_CFG = AssociativeMemoryConfig(use_fft=True, use_attention=False, correctness_z_dim=CORRECTNESS_Z_DIM)

if FAST_MODE:
    SEEDS = (42,)
    EPOCHS, MAX_TRAIN, MAX_CAL, MAX_TEST, MAX_EXT = 2, 800, 200, 400, 400
else:
    EPOCHS = 8
    MAX_TRAIN, MAX_CAL, MAX_TEST = DERMA_TRAIN, DERMA_CAL, DERMA_TEST
    MAX_EXT = DERMA_TEST

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROBE_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)

print("TASKS", TASKS)
print("OUTPUT_DIR", OUTPUT_DIR)
print(f"CORRECTNESS_Z_DIM={CORRECTNESS_Z_DIM}  z_combined dim={CORRECTNESS_Z_DIM * NUM_LEVELS}")
print(f"caps: train={MAX_TRAIN} cal={MAX_CAL} test={MAX_TEST} probe_n={TEST_SAMPLE_SIZE}")
print(f"RUN_BUILD={RUN_BUILD}  FORCE_RETRAIN={FORCE_RETRAIN}  FORCE_REDEPLOY={FORCE_REDEPLOY}")

TASKS ('bloodmnist', 'organcmnist')
OUTPUT_DIR C:\Users\Sounak Sinha\OneDrive\Desktop\Research\optimization-history-failure-detection\research\correctness_memory_fft\cross_dataset
CORRECTNESS_Z_DIM=7  z_combined dim=28
caps: train=5000 cal=1602 test=2000 probe_n=2000
RUN_BUILD=False  FORCE_RETRAIN=False  FORCE_REDEPLOY=False


## Probe helpers

Correctness memory deploy features: $z_j \in \mathbb{R}^{7}$ per level, $z_{\mathrm{combined}} \in \mathbb{R}^{28}$.

In [3]:
def infer_z_dim(df: pd.DataFrame) -> int:
    cols = [c for c in df.columns if c.startswith("z1_d")]
    if not cols:
        raise ValueError("Feature cache missing z1_d* columns; redeploy Part II.")
    return len(cols)


def feature_cache_paths(task: str, seed: int) -> tuple[Path, Path]:
    seed_dir = FEATURE_ROOT / task / f"seed{seed}"
    return seed_dir / "cal_features.csv", seed_dir / "test_features_full.csv"


def subsample_test_df(test_full: pd.DataFrame, *, sample_size: int | None, seed: int) -> pd.DataFrame:
    if sample_size is None or sample_size >= len(test_full):
        return test_full.reset_index(drop=True)
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.choice(len(test_full), size=int(sample_size), replace=False))
    return test_full.iloc[idx].reset_index(drop=True)


def fit_l2_probe_on_cal(cal_df: pd.DataFrame, feature_cols: list[str], *, representation: str) -> dict[str, Any]:
    req = feature_cols + ["error"]
    mask = cal_df[req].notna().all(axis=1).to_numpy()
    x_cal = cal_df.loc[mask, feature_cols].to_numpy(dtype=np.float64)
    y_cal = cal_df.loc[mask, "error"].to_numpy(dtype=int)
    if len(x_cal) < 10 or len(np.unique(y_cal)) < 2:
        raise ValueError(f"Invalid calibration data for {representation}: n={len(x_cal)}")
    pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(C=LOGISTIC_C, max_iter=5000, random_state=42)),
        ]
    )
    pipe.fit(x_cal, y_cal)
    cal_scores = pipe.predict_proba(x_cal)[:, 1]
    return {
        "representation": representation,
        "feature_cols": feature_cols,
        "n_cal": int(len(x_cal)),
        "intercept": float(pipe.named_steps["clf"].intercept_[0]),
        "pipeline": pipe,
        "cal_auroc": float(roc_auc_score(y_cal, cal_scores)),
        "cal_auprc": float(average_precision_score(y_cal, cal_scores)),
    }


def eval_l2_probe(fitted: dict[str, Any], df: pd.DataFrame) -> dict[str, Any]:
    feature_cols = fitted["feature_cols"]
    req = feature_cols + ["error"]
    mask = df[req].notna().all(axis=1).to_numpy()
    x = df.loc[mask, feature_cols].to_numpy(dtype=np.float64)
    y = df.loc[mask, "error"].to_numpy(dtype=int)
    scores = fitted["pipeline"].predict_proba(x)[:, 1]
    return {
        **fitted,
        "n_test": int(len(y)),
        "auroc": float(roc_auc_score(y, scores)),
        "auprc": float(average_precision_score(y, scores)),
    }


def eval_entropy_baseline(test_df: pd.DataFrame) -> dict[str, float]:
    mask = test_df[["normalized_entropy", "error"]].notna().all(axis=1).to_numpy()
    y = test_df.loc[mask, "error"].to_numpy(dtype=int)
    h = test_df.loc[mask, "normalized_entropy"].to_numpy(dtype=float)
    if len(y) < 10 or len(np.unique(y)) < 2:
        return {"n_test": int(len(y)), "auroc": float("nan"), "auprc": float("nan")}
    return {
        "n_test": int(len(y)),
        "auroc": float(roc_auc_score(y, h)),
        "auprc": float(average_precision_score(y, h)),
    }

## 1. Train (Part I — per task)

Mirrors `dermamnist_correctness_memory_full.ipynb` Part I. Skips seeds with existing correctness artifacts unless `FORCE_RETRAIN=True`.

In [4]:
bundles: dict[str, Any] = {}
train_summaries: list[dict[str, Any]] = []

train_cfg = ClinicalTrainingConfig(
    seeds=SEEDS,
    epochs=EPOCHS,
    output_dir=OUTPUT_DIR,
    verbose=1,
    associative_memory=ASSOC_CFG,
)

for task in TASKS:
    ds_cfg = ClinicalDatasetConfig(
        task=task,
        data_dir=DATA_DIR,
        max_train=MAX_TRAIN,
        max_cal=MAX_CAL,
        max_test=MAX_TEST,
        max_external=MAX_EXT,
    )
    bundle = load_clinical_bundle(ds_cfg)
    bundles[task] = bundle
    print(f"{task}: n_train={len(bundle.x_train)} n_cal={len(bundle.x_cal)} n_test={len(bundle.x_test)} C={bundle.num_classes}")

    if not RUN_BUILD:
        missing = [s for s in SEEDS if not correctness_artifacts_exist(OUTPUT_DIR, task, s)]
        if not missing:
            print(f"  RUN_BUILD=False — using cached artifacts for {task}")
            continue
        print(f"  RUN_BUILD=False — training missing seeds only: {missing}")
        seeds_to_train = missing
    else:
        seeds_to_train = list(SEEDS)

    for seed in seeds_to_train:
        ckpt = checkpoint_path(OUTPUT_DIR, task, seed)
        if FORCE_RETRAIN:
            for path in (
                ckpt,
                correctness_memory_path(OUTPUT_DIR, task, seed),
                correctness_checkpoint_bundle_path(OUTPUT_DIR, task, seed),
            ):
                path.unlink(missing_ok=True)
            seed_cache = FEATURE_ROOT / task / f"seed{seed}"
            if seed_cache.exists():
                shutil.rmtree(seed_cache)
        elif correctness_memory_exists(OUTPUT_DIR, task, seed):
            recover_correctness_checkpoint_bundle(OUTPUT_DIR, task, seed)
        if not FORCE_RETRAIN and ckpt.exists() and correctness_artifacts_exist(OUTPUT_DIR, task, seed):
            print(f"  skip {task} seed={seed} (checkpoint + correctness artifacts exist)")
            continue
        params, opt_state, _, _, summary = train_one_seed_correctness(bundle, seed=seed, cfg=train_cfg)
        save_checkpoint(ckpt, params=params, opt_state=opt_state, task=task, seed=seed, cfg=train_cfg, summary=summary)
        train_summaries.append(summary)
        print(
            f"  trained {task} seed={seed} test_acc={summary.get('test_acc'):.3f} "
            f"z_dim={summary.get('correctness_z_dim')}"
        )

bloodmnist: n_train=5000 n_cal=1602 n_test=2000 C=8
  RUN_BUILD=False — using cached artifacts for bloodmnist
organcmnist: n_train=5000 n_cal=1602 n_test=2000 C=11
  RUN_BUILD=False — training missing seeds only: [456]
  [organcmnist seed=456] correctness-memory training: 5000 samples, 8 epochs, ~79 batches/epoch
  [organcmnist seed=456] epoch 1/8: train_loss=0.8070 cal_acc=0.808 test_acc=0.738 (1072.2s)
  [organcmnist seed=456] epoch 2/8: train_loss=0.4382 cal_acc=0.893 test_acc=0.815 (667.2s)
  [organcmnist seed=456] epoch 3/8: train_loss=0.3104 cal_acc=0.905 test_acc=0.841 (467.5s)
  [organcmnist seed=456] epoch 4/8: train_loss=0.2678 cal_acc=0.925 test_acc=0.854 (1052.7s)
  [organcmnist seed=456] epoch 5/8: train_loss=0.2050 cal_acc=0.939 test_acc=0.870 (702.4s)
  [organcmnist seed=456] epoch 6/8: train_loss=0.1416 cal_acc=0.937 test_acc=0.874 (699.4s)
  [organcmnist seed=456] epoch 7/8: train_loss=0.1432 cal_acc=0.936 test_acc=0.858 (720.3s)
  [organcmnist seed=456] epoch 8/8: tra

## 2. Deploy features + $z_{\mathrm{combined}}$ vs $H$

Label-free correctness-memory deployment on cal/test; logistic probe fit on cal-only $z_{\mathrm{combined}} \in \mathbb{R}^{28}$.

In [5]:
def deploy_split_for_seed(
    task: str,
    seed: int,
    x: np.ndarray,
    y: np.ndarray,
    ids: np.ndarray,
    *,
    num_classes: int,
) -> pd.DataFrame:
    if not correctness_artifacts_exist(OUTPUT_DIR, task, seed):
        raise FileNotFoundError(f"Missing correctness artifacts for {task} seed={seed}")
    with open(checkpoint_path(OUTPUT_DIR, task, seed), "rb") as f:
        params = pickle.load(f)["params"]
    mem_state, _, _ = load_correctness_artifacts(OUTPUT_DIR, task, seed)
    _, df = run_correctness_deployment_on_split(
        params,
        x,
        y,
        ids,
        mem_state,
        num_classes=num_classes,
    )
    return df


def load_or_deploy_features(task: str, bundle: Any, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    cal_path, test_path = feature_cache_paths(task, seed)
    if not FORCE_REDEPLOY and cal_path.exists() and test_path.exists():
        cal_df = pd.read_csv(cal_path)
        test_df = pd.read_csv(test_path)
        z_dim = infer_z_dim(cal_df)
        if z_dim != CORRECTNESS_Z_DIM:
            print(f"  stale cache {task} seed={seed} (z_dim={z_dim}); redeploying")
        else:
            return cal_df, test_df

    print(f"  deploying {task} seed={seed} ...", flush=True)
    cal_df = deploy_split_for_seed(
        task,
        seed,
        bundle.x_cal,
        bundle.y_cal,
        bundle.sample_ids["cal"],
        num_classes=bundle.num_classes,
    )
    test_full = deploy_split_for_seed(
        task,
        seed,
        bundle.x_test,
        bundle.y_test,
        bundle.sample_ids["test"],
        num_classes=bundle.num_classes,
    )
    cal_path.parent.mkdir(parents=True, exist_ok=True)
    cal_df.to_csv(cal_path, index=False)
    test_full.to_csv(test_path, index=False)
    return cal_df, test_full


phase3_rows: list[dict[str, Any]] = []
task_summaries: dict[str, dict[str, Any]] = {}

for task in TASKS:
    bundle = bundles[task]
    cal_frames: dict[int, pd.DataFrame] = {}
    test_frames: dict[int, pd.DataFrame] = {}

    for seed in SEEDS:
        cal_df, test_full = load_or_deploy_features(task, bundle, seed)
        cal_frames[seed] = cal_df
        test_frames[seed] = subsample_test_df(
            test_full,
            sample_size=TEST_SAMPLE_SIZE,
            seed=RANDOM_SEED + seed,
        )

    z_dim = infer_z_dim(cal_frames[SEEDS[0]])
    z_combined_feature_cols = z_combined_cols(NUM_LEVELS, z_dim=z_dim)

    for seed in SEEDS:
        cal_df = cal_frames[seed]
        test_df = test_frames[seed]
        z_fit = fit_l2_probe_on_cal(cal_df, z_combined_feature_cols, representation="z_combined")
        z_test = eval_l2_probe(z_fit, test_df)
        h_test = eval_entropy_baseline(test_df)
        phase3_rows.append(
            {
                "task": task,
                "seed": seed,
                "model": "entropy_H",
                "representation": "z_combined",
                "n_cal": z_fit["n_cal"],
                "n_test": h_test["n_test"],
                "auroc": h_test["auroc"],
                "auprc": h_test["auprc"],
            }
        )
        phase3_rows.append(
            {
                "task": task,
                "seed": seed,
                "model": "z_combined",
                "representation": "z_combined",
                "n_cal": z_fit["n_cal"],
                "n_test": z_test["n_test"],
                "auroc": z_test["auroc"],
                "auprc": z_test["auprc"],
            }
        )
        phase3_rows.append(
            {
                "task": task,
                "seed": seed,
                "model": "delta_z_combined_minus_H",
                "representation": "z_combined",
                "n_cal": z_fit["n_cal"],
                "n_test": z_test["n_test"],
                "auroc": float(z_test["auroc"] - h_test["auroc"]),
                "auprc": float(z_test["auprc"] - h_test["auprc"]),
            }
        )

    task_phase3 = pd.DataFrame([r for r in phase3_rows if r["task"] == task])
    h_auroc_mean = float(task_phase3.loc[task_phase3["model"] == "entropy_H", "auroc"].mean())
    z_auroc_mean = float(task_phase3.loc[task_phase3["model"] == "z_combined", "auroc"].mean())
    h_auprc_mean = float(task_phase3.loc[task_phase3["model"] == "entropy_H", "auprc"].mean())
    z_auprc_mean = float(task_phase3.loc[task_phase3["model"] == "z_combined", "auprc"].mean())
    task_summaries[task] = {
        "task": task,
        "num_classes": bundle.num_classes,
        "correctness_z_dim": z_dim,
        "z_combined_dim": z_dim * NUM_LEVELS,
        "mean_auroc_H": h_auroc_mean,
        "mean_auroc_z_combined": z_auroc_mean,
        "mean_delta_auroc": float(z_auroc_mean - h_auroc_mean),
        "mean_auprc_H": h_auprc_mean,
        "mean_auprc_z_combined": z_auprc_mean,
        "mean_delta_auprc": float(z_auprc_mean - h_auprc_mean),
        "z_combined_beats_H_on_auroc": z_auroc_mean > h_auroc_mean,
        "z_combined_beats_H_on_auprc": z_auprc_mean > h_auprc_mean,
    }

phase3_df = pd.DataFrame(phase3_rows)
summary_df = pd.DataFrame(task_summaries.values())

phase3_df.to_csv(PROBE_DIR / "phase3_per_task_seed.csv", index=False)
summary_df.to_csv(PROBE_DIR / "cross_dataset_probe_summary.csv", index=False)

results = {
    "tasks": list(TASKS),
    "seeds": list(SEEDS),
    "memory_kind": "correctness_hd",
    "correctness_z_dim": CORRECTNESS_Z_DIM,
    "test_sample_size": TEST_SAMPLE_SIZE,
    "task_summaries": task_summaries,
}
(PROBE_DIR / "cross_dataset_probe_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

display(summary_df.round(4))
display(
    phase3_df.pivot_table(
        index=["task", "seed"],
        columns="model",
        values=["auroc", "auprc"],
    ).round(4)
)

  deploying organcmnist seed=42 ...
  deploying organcmnist seed=123 ...
  deploying organcmnist seed=456 ...


,task,num_classes,correctness_z_dim,z_combined_dim,mean_auroc_H,mean_auroc_z_combined,mean_delta_auroc,mean_auprc_H,mean_auprc_z_combined,mean_delta_auprc,z_combined_beats_H_on_auroc,z_combined_beats_H_on_auprc
0,bloodmnist,8,7,28,0.7656,0.7975,0.0319,0.9320,0.9482,0.0162,True,True
1,organcmnist,11,7,28,0.7433,0.8218,0.0785,0.8787,0.9134,0.0347,True,True


auprc                       \
model            delta_z_combined_minus_H entropy_H z_combined   
task        seed                                                 
bloodmnist  42                     0.0123    0.9158     0.9281   
            123                    0.0208    0.9381     0.9590   
            456                    0.0154    0.9421     0.9576   
organcmnist 42                     0.0715    0.8999     0.9714   
            123                   -0.0066    0.8623     0.8558   
            456                    0.0392    0.8738     0.9130   

                                    auroc                       
model            delta_z_combined_minus_H entropy_H z_combined  
task        seed                                                
bloodmnist  42                     0.0239    0.7273     0.7512  
            123                    0.0272    0.8018     0.8290  
            456                    0.0445    0.7678     0.8123  
organcmnist 42                     0.1641    0.7501     0.9142  
            123                   -0.0034    0.7249     0.7215  
            456                    0.0748    0.7549     0.8297

## 2b. $z_{\mathrm{combined}}$ vs $h_T$ probe

Cal-fit logistic on 512-d penultimate features $h_T(x)$ (single-sample forward, matching deployment). Cached as `probe_features/<task>/seed*/{cal,test}_ht.npy`.

In [7]:
# z_combined vs h_T probe (512-d penultimate logistic, single-sample forward)
import time

import jax.numpy as jnp

from research.common.resnet import resnet18_features
from research.correctness_memory_fft.probe_experiments.correctness_probe_analysis import HT_COLS, HT_DIM

FORCE_HT = False


def _id_index(ids: np.ndarray) -> dict[str, int]:
    return {str(s): int(i) for i, s in enumerate(ids)}


def extract_ht_matrix(
    params: dict[str, Any],
    x_split: np.ndarray,
    id_to_idx: dict[str, int],
    sample_ids: pd.Series,
) -> np.ndarray:
    rows: list[np.ndarray] = []
    for sid in sample_ids.astype(str):
        i = id_to_idx[str(sid)]
        h = np.asarray(
            resnet18_features(params, jnp.asarray(x_split[i], dtype=jnp.float32)),
            dtype=np.float32,
        ).reshape(-1)
        rows.append(h)
    return np.stack(rows, axis=0)


def attach_ht(df: pd.DataFrame, ht: np.ndarray) -> pd.DataFrame:
    ht_df = pd.DataFrame(ht, columns=HT_COLS, index=df.index)
    return pd.concat([df.reset_index(drop=True), ht_df], axis=1)


def ht_cache_paths(task: str, seed: int, split: str) -> tuple[Path, Path]:
    cache_dir = FEATURE_ROOT / task / f"seed{seed}"
    return cache_dir / f"{split}_ht.npy", cache_dir / f"{split}_sample_ids.npy"


def load_or_extract_ht(
    task: str,
    seed: int,
    split: str,
    df: pd.DataFrame,
    *,
    bundle: Any,
    force: bool = False,
) -> pd.DataFrame:
    cache_path, ids_path = ht_cache_paths(task, seed, split)
    sample_ids = df["sample_id"].astype(str).to_numpy()
    if cache_path.exists() and ids_path.exists() and not force:
        cached_ids = np.load(ids_path, allow_pickle=True)
        if np.array_equal(cached_ids, sample_ids):
            return attach_ht(df, np.load(cache_path))

    with open(checkpoint_path(OUTPUT_DIR, task, seed), "rb") as f:
        params = pickle.load(f)["params"]
    x_split = bundle.x_cal if split == "cal" else bundle.x_test
    id_to_idx = _id_index(bundle.sample_ids[split])
    t0 = time.perf_counter()
    ht = extract_ht_matrix(params, x_split, id_to_idx, df["sample_id"])
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, ht)
    np.save(ids_path, sample_ids)
    print(f"    extracted h_T {task} seed={seed} {split} n={len(df)} in {time.perf_counter() - t0:.1f}s")
    return attach_ht(df, ht)


ht_probe_rows: list[dict[str, Any]] = []
ht_task_summaries: dict[str, dict[str, Any]] = {}

for task in TASKS:
    bundle = bundles[task]
    cal_path0, _ = feature_cache_paths(task, SEEDS[0])
    z_dim = infer_z_dim(pd.read_csv(cal_path0, nrows=5))
    z_feature_cols = z_combined_cols(NUM_LEVELS, z_dim=z_dim)

    for seed in SEEDS:
        cal_path, test_path = feature_cache_paths(task, seed)
        cal_df = load_or_extract_ht(task, seed, "cal", pd.read_csv(cal_path), bundle=bundle, force=FORCE_HT)
        test_full = load_or_extract_ht(
            task, seed, "test", pd.read_csv(test_path), bundle=bundle, force=FORCE_HT
        )
        test_df = subsample_test_df(
            test_full,
            sample_size=TEST_SAMPLE_SIZE,
            seed=RANDOM_SEED + seed,
        )

        z_fit = fit_l2_probe_on_cal(cal_df, z_feature_cols, representation="z_combined")
        z_test = eval_l2_probe(z_fit, test_df)
        ht_fit = fit_l2_probe_on_cal(cal_df, HT_COLS, representation="h_T")
        ht_test = eval_l2_probe(ht_fit, test_df)

        ht_probe_rows.extend(
            [
                {
                    "task": task,
                    "seed": seed,
                    "model": "h_T_probe",
                    "representation": "h_T",
                    "n_cal": ht_fit["n_cal"],
                    "n_test": ht_test["n_test"],
                    "cal_auroc": ht_fit["cal_auroc"],
                    "auroc": ht_test["auroc"],
                    "auprc": ht_test["auprc"],
                },
                {
                    "task": task,
                    "seed": seed,
                    "model": "z_combined",
                    "representation": "z_combined",
                    "n_cal": z_fit["n_cal"],
                    "n_test": z_test["n_test"],
                    "cal_auroc": z_fit["cal_auroc"],
                    "auroc": z_test["auroc"],
                    "auprc": z_test["auprc"],
                },
                {
                    "task": task,
                    "seed": seed,
                    "model": "delta_z_combined_minus_h_T",
                    "representation": "z_combined",
                    "n_cal": z_fit["n_cal"],
                    "n_test": z_test["n_test"],
                    "cal_auroc": float("nan"),
                    "auroc": float(z_test["auroc"] - ht_test["auroc"]),
                    "auprc": float(z_test["auprc"] - ht_test["auprc"]),
                },
            ]
        )

    task_ht = pd.DataFrame([r for r in ht_probe_rows if r["task"] == task])
    z_mean = float(task_ht.loc[task_ht["model"] == "z_combined", "auroc"].mean())
    ht_mean = float(task_ht.loc[task_ht["model"] == "h_T_probe", "auroc"].mean())
    z_auprc = float(task_ht.loc[task_ht["model"] == "z_combined", "auprc"].mean())
    ht_auprc = float(task_ht.loc[task_ht["model"] == "h_T_probe", "auprc"].mean())
    ht_task_summaries[task] = {
        "task": task,
        "ht_dim": HT_DIM,
        "z_combined_dim": z_dim * NUM_LEVELS,
        "mean_auroc_z_combined": z_mean,
        "mean_auroc_h_T_probe": ht_mean,
        "mean_delta_z_minus_h_T": float(z_mean - ht_mean),
        "mean_auprc_z_combined": z_auprc,
        "mean_auprc_h_T_probe": ht_auprc,
        "mean_delta_auprc_z_minus_h_T": float(z_auprc - ht_auprc),
        "z_combined_beats_h_T_on_auroc": z_mean > ht_mean,
        "z_combined_beats_h_T_on_auprc": z_auprc > ht_auprc,
    }

ht_probe_df = pd.DataFrame(ht_probe_rows)
ht_summary_df = pd.DataFrame(ht_task_summaries.values())
ht_probe_df.to_csv(PROBE_DIR / "ht_probe_per_task_seed.csv", index=False)
ht_summary_df.to_csv(PROBE_DIR / "ht_probe_cross_dataset_summary.csv", index=False)

print(f"h_T dim={HT_DIM}  z_combined dim={CORRECTNESS_Z_DIM * NUM_LEVELS}")
for task, s in ht_task_summaries.items():
    print(
        f"{task}: AUROC z_combined={s['mean_auroc_z_combined']:.4f}  "
        f"h_T={s['mean_auroc_h_T_probe']:.4f}  "
        f"z-h_T={s['mean_delta_z_minus_h_T']:+.4f}"
    )

display(ht_summary_df.round(4))
display(
    ht_probe_df.pivot_table(
        index=["task", "seed"],
        columns="model",
        values=["auroc", "auprc"],
    ).round(4)
)

    extracted h_T bloodmnist seed=42 cal n=1602 in 29.4s
    extracted h_T bloodmnist seed=42 test n=2000 in 38.3s
    extracted h_T bloodmnist seed=123 cal n=1602 in 31.8s
    extracted h_T bloodmnist seed=123 test n=2000 in 37.2s
    extracted h_T bloodmnist seed=456 cal n=1602 in 29.3s
    extracted h_T bloodmnist seed=456 test n=2000 in 40.8s
    extracted h_T organcmnist seed=42 cal n=1602 in 35.8s
    extracted h_T organcmnist seed=42 test n=2000 in 40.2s
    extracted h_T organcmnist seed=123 cal n=1602 in 32.1s
    extracted h_T organcmnist seed=123 test n=2000 in 43.5s
    extracted h_T organcmnist seed=456 cal n=1602 in 89.8s
    extracted h_T organcmnist seed=456 test n=2000 in 146.2s
h_T dim=512  z_combined dim=28
bloodmnist: AUROC z_combined=0.7975  h_T=0.8185  z-h_T=-0.0210
organcmnist: AUROC z_combined=0.8218  h_T=0.8926  z-h_T=-0.0708


,task,ht_dim,z_combined_dim,mean_auroc_z_combined,mean_auroc_h_T_probe,mean_delta_z_minus_h_T,mean_auprc_z_combined,mean_auprc_h_T_probe,mean_delta_auprc_z_minus_h_T,z_combined_beats_h_T_on_auroc,z_combined_beats_h_T_on_auprc
0,bloodmnist,512,28,0.7975,0.8185,-0.0210,0.9482,0.9528,-0.0046,False,False
1,organcmnist,512,28,0.8218,0.8926,-0.0708,0.9134,0.9518,-0.0384,False,False


auprc                       \
model            delta_z_combined_minus_h_T h_T_probe z_combined   
task        seed                                                   
bloodmnist  42                      -0.0152    0.9433     0.9281   
            123                     -0.0006    0.9596     0.9590   
            456                      0.0019    0.9557     0.9576   
organcmnist 42                      -0.0131    0.9844     0.9714   
            123                     -0.0646    0.9204     0.8558   
            456                     -0.0375    0.9505     0.9130   

                                      auroc                       
model            delta_z_combined_minus_h_T h_T_probe z_combined  
task        seed                                                  
bloodmnist  42                      -0.0594    0.8106     0.7512  
            123                      0.0013    0.8277     0.8290  
            456                     -0.0049    0.8172     0.8123  
organcmnist 42                      -0.0381    0.9523     0.9142  
            123                     -0.1035    0.8250     0.7215  
            456                     -0.0707    0.9004     0.8297

## 2c. $z_{\mathrm{combined}} + h_T$ probe

Cal-fit logistic on concatenated features $[z_{\mathrm{combined}}(x); h_T(x)] \in \mathbb{R}^{540}$, compared with each component alone (requires Section 2b h_T cache).

In [8]:
# z_combined + h_T combined probe ([z_combined; h_T] in R^{28+512})
from research.correctness_memory_fft.probe_experiments.correctness_probe_analysis import HT_COLS, HT_DIM

if "load_or_extract_ht" not in globals():
    raise RuntimeError("Run Section 2b first (defines load_or_extract_ht and FORCE_HT).")

combined_probe_rows: list[dict[str, Any]] = []
combined_task_summaries: dict[str, dict[str, Any]] = {}

for task in TASKS:
    bundle = bundles[task]
    cal_path0, _ = feature_cache_paths(task, SEEDS[0])
    z_dim = infer_z_dim(pd.read_csv(cal_path0, nrows=5))
    z_feature_cols = z_combined_cols(NUM_LEVELS, z_dim=z_dim)
    combined_feature_cols = z_feature_cols + HT_COLS

    for seed in SEEDS:
        cal_path, test_path = feature_cache_paths(task, seed)
        cal_df = load_or_extract_ht(
            task, seed, "cal", pd.read_csv(cal_path), bundle=bundle, force=FORCE_HT
        )
        test_full = load_or_extract_ht(
            task, seed, "test", pd.read_csv(test_path), bundle=bundle, force=FORCE_HT
        )
        test_df = subsample_test_df(
            test_full,
            sample_size=TEST_SAMPLE_SIZE,
            seed=RANDOM_SEED + seed,
        )

        z_fit = fit_l2_probe_on_cal(cal_df, z_feature_cols, representation="z_combined")
        z_test = eval_l2_probe(z_fit, test_df)
        ht_fit = fit_l2_probe_on_cal(cal_df, HT_COLS, representation="h_T")
        ht_test = eval_l2_probe(ht_fit, test_df)
        combined_fit = fit_l2_probe_on_cal(
            cal_df, combined_feature_cols, representation="z_combined_plus_h_T"
        )
        combined_test = eval_l2_probe(combined_fit, test_df)

        for model, fit, test in (
            ("z_combined", z_fit, z_test),
            ("h_T_probe", ht_fit, ht_test),
            ("z_combined_plus_h_T", combined_fit, combined_test),
        ):
            combined_probe_rows.append(
                {
                    "task": task,
                    "seed": seed,
                    "model": model,
                    "representation": model,
                    "n_cal": fit["n_cal"],
                    "n_test": test["n_test"],
                    "cal_auroc": fit["cal_auroc"],
                    "auroc": test["auroc"],
                    "auprc": test["auprc"],
                    "feature_dim": len(fit["feature_cols"]),
                }
            )

        combined_probe_rows.append(
            {
                "task": task,
                "seed": seed,
                "model": "delta_combined_minus_z",
                "representation": "z_combined_plus_h_T",
                "n_cal": combined_fit["n_cal"],
                "n_test": combined_test["n_test"],
                "cal_auroc": float("nan"),
                "auroc": float(combined_test["auroc"] - z_test["auroc"]),
                "auprc": float(combined_test["auprc"] - z_test["auprc"]),
                "feature_dim": len(combined_feature_cols),
            }
        )
        combined_probe_rows.append(
            {
                "task": task,
                "seed": seed,
                "model": "delta_combined_minus_h_T",
                "representation": "z_combined_plus_h_T",
                "n_cal": combined_fit["n_cal"],
                "n_test": combined_test["n_test"],
                "cal_auroc": float("nan"),
                "auroc": float(combined_test["auroc"] - ht_test["auroc"]),
                "auprc": float(combined_test["auprc"] - ht_test["auprc"]),
                "feature_dim": len(combined_feature_cols),
            }
        )

    task_df = pd.DataFrame([r for r in combined_probe_rows if r["task"] == task])
    z_mean = float(task_df.loc[task_df["model"] == "z_combined", "auroc"].mean())
    ht_mean = float(task_df.loc[task_df["model"] == "h_T_probe", "auroc"].mean())
    comb_mean = float(task_df.loc[task_df["model"] == "z_combined_plus_h_T", "auroc"].mean())
    z_auprc = float(task_df.loc[task_df["model"] == "z_combined", "auprc"].mean())
    ht_auprc = float(task_df.loc[task_df["model"] == "h_T_probe", "auprc"].mean())
    comb_auprc = float(task_df.loc[task_df["model"] == "z_combined_plus_h_T", "auprc"].mean())
    combined_task_summaries[task] = {
        "task": task,
        "z_combined_dim": z_dim * NUM_LEVELS,
        "ht_dim": HT_DIM,
        "combined_dim": len(combined_feature_cols),
        "mean_auroc_z_combined": z_mean,
        "mean_auroc_h_T_probe": ht_mean,
        "mean_auroc_z_combined_plus_h_T": comb_mean,
        "mean_delta_combined_minus_z": float(comb_mean - z_mean),
        "mean_delta_combined_minus_h_T": float(comb_mean - ht_mean),
        "mean_auprc_z_combined": z_auprc,
        "mean_auprc_h_T_probe": ht_auprc,
        "mean_auprc_z_combined_plus_h_T": comb_auprc,
        "combined_beats_z_on_auroc": comb_mean > z_mean,
        "combined_beats_h_T_on_auroc": comb_mean > ht_mean,
    }

combined_probe_df = pd.DataFrame(combined_probe_rows)
combined_summary_df = pd.DataFrame(combined_task_summaries.values())
combined_probe_df.to_csv(PROBE_DIR / "z_combined_plus_ht_per_task_seed.csv", index=False)
combined_summary_df.to_csv(PROBE_DIR / "z_combined_plus_ht_cross_dataset_summary.csv", index=False)

print(f"combined feature dim={CORRECTNESS_Z_DIM * NUM_LEVELS + HT_DIM}  (z={CORRECTNESS_Z_DIM * NUM_LEVELS} + h_T={HT_DIM})")
for task, s in combined_task_summaries.items():
    print(
        f"{task}: AUROC z={s['mean_auroc_z_combined']:.4f}  "
        f"h_T={s['mean_auroc_h_T_probe']:.4f}  "
        f"z+h_T={s['mean_auroc_z_combined_plus_h_T']:.4f}  "
        f"(+z {s['mean_delta_combined_minus_z']:+.4f}, +h_T {s['mean_delta_combined_minus_h_T']:+.4f})"
    )

display(combined_summary_df.round(4))
display(
    combined_probe_df.pivot_table(
        index=["task", "seed"],
        columns="model",
        values=["auroc", "auprc"],
    ).round(4)
)

combined feature dim=540  (z=28 + h_T=512)
bloodmnist: AUROC z=0.7975  h_T=0.8185  z+h_T=0.8199  (+z +0.0224, +h_T +0.0014)
organcmnist: AUROC z=0.8218  h_T=0.8926  z+h_T=0.8931  (+z +0.0713, +h_T +0.0005)


,task,z_combined_dim,ht_dim,combined_dim,mean_auroc_z_combined,mean_auroc_h_T_probe,mean_auroc_z_combined_plus_h_T,mean_delta_combined_minus_z,mean_delta_combined_minus_h_T,mean_auprc_z_combined,mean_auprc_h_T_probe,mean_auprc_z_combined_plus_h_T,combined_beats_z_on_auroc,combined_beats_h_T_on_auroc
0,bloodmnist,28,512,540,0.7975,0.8185,0.8199,0.0224,0.0014,0.9482,0.9528,0.9533,True,True
1,organcmnist,28,512,540,0.8218,0.8926,0.8931,0.0713,0.0005,0.9134,0.9518,0.9523,True,True


auprc                                   \
model            delta_combined_minus_h_T delta_combined_minus_z h_T_probe   
task        seed                                                             
bloodmnist  42                     0.0008                 0.0160    0.9433   
            123                    0.0003                 0.0009    0.9596   
            456                    0.0004                -0.0015    0.9557   
organcmnist 42                    -0.0001                 0.0130    0.9844   
            123                    0.0012                 0.0659    0.9204   
            456                    0.0003                 0.0378    0.9505   

                                                                   auroc  \
model            z_combined z_combined_plus_h_T delta_combined_minus_h_T   
task        seed                                                           
bloodmnist  42       0.9281              0.9441                   0.0022   
            123      0.9590              0.9599                   0.0002   
            456      0.9576              0.9561                   0.0017   
organcmnist 42       0.9714              0.9843                   0.0000   
            123      0.8558              0.9216                   0.0014   
            456      0.9130              0.9508                   0.0002   

                                                              \
model            delta_combined_minus_z h_T_probe z_combined   
task        seed                                               
bloodmnist  42                   0.0616    0.8106     0.7512   
            123                 -0.0011    0.8277     0.8290   
            456                  0.0066    0.8172     0.8123   
organcmnist 42                   0.0381    0.9523     0.9142   
            123                  0.1049    0.8250     0.7215   
            456                  0.0709    0.9004     0.8297   

                                      
model            z_combined_plus_h_T  
task        seed                      
bloodmnist  42                0.8128  
            123               0.8279  
            456               0.8189  
organcmnist 42                0.9523  
            123               0.8264  
            456               0.9006

## 3. Summary

In [6]:
summary_path = PROBE_DIR / "cross_dataset_probe_summary.md"
lines = [
    "# BloodMNIST + OrganCMNIST — correctness memory $z_{\\mathrm{HD}}$ vs $H$",
    "",
    f"Correctness memory: $z_j \\in \\mathbb{{R}}^{{{CORRECTNESS_Z_DIM}}}$, "
    f"$z_{{\\mathrm{{HD}}}} \\in \\mathbb{{R}}^{{{CORRECTNESS_Z_DIM * NUM_LEVELS}}}$.",
    "",
]
for task, s in task_summaries.items():
    lines.append(
        f"- **{task}** ($z_{{\\mathrm{{HD}}}}$ dim={s['z_combined_dim']}): "
        f"AUROC $z_{{\\mathrm{{combined}}}}$={s['mean_auroc_z_combined']:.4f} vs $H$={s['mean_auroc_H']:.4f} "
        f"($\\Delta$={s['mean_delta_auroc']:+.4f}); "
        f"AUPRC {s['mean_auprc_z_combined']:.4f} vs {s['mean_auprc_H']:.4f} "
        f"($\\Delta$={s['mean_delta_auprc']:+.4f})"
    )
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(summary_path.read_text(encoding="utf-8"))

# BloodMNIST + OrganCMNIST — correctness memory $z_{\mathrm{HD}}$ vs $H$

Correctness memory: $z_j \in \mathbb{R}^{7}$, $z_{\mathrm{HD}} \in \mathbb{R}^{28}$.

- **bloodmnist** ($z_{\mathrm{HD}}$ dim=28): AUROC $z_{\mathrm{combined}}$=0.7975 vs $H$=0.7656 ($\Delta$=+0.0319); AUPRC 0.9482 vs 0.9320 ($\Delta$=+0.0162)
- **organcmnist** ($z_{\mathrm{HD}}$ dim=28): AUROC $z_{\mathrm{combined}}$=0.8218 vs $H$=0.7433 ($\Delta$=+0.0785); AUPRC 0.9134 vs 0.8787 ($\Delta$=+0.0347)

